In [4]:
# ================================================================
# TORCH: Factory-Level Risk Index — Myanmar Apparel Supply Chains
# Stage 4: ML Modeling — Multi-Label Classification
# ================================================================

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import classification_report, hamming_loss
import joblib

news  = pd.read_csv("stage3_news_features.csv")
bhrrc = pd.read_csv("stage1_bhrrc_labels.csv")

# ----------------------------------------------------------------
# 1. Prepare the labelled dataset
# ----------------------------------------------------------------

news["factory_ref"] = news["osh_factory_matched"].fillna(news["bhrrc_factory_matched"])
labelled = news[news["factory_ref"].notna()].copy()

label_cols = [
    "label_child_labour", "label_forced_labour", "label_discrimination",
    "label_freedom_of_association", "label_working_hours",
    "label_compensation", "label_osh", "label_contracts"
]

bhrrc_labels = bhrrc[["factory_name"] + label_cols].copy()
bhrrc_labels["factory_lower"] = bhrrc_labels["factory_name"].str.lower().str.strip()
labelled["factory_ref_lower"] = labelled["factory_ref"].str.lower().str.strip()

labelled = labelled.merge(
    bhrrc_labels.drop(columns=["factory_name"]),
    left_on="factory_ref_lower",
    right_on="factory_lower",
    how="inner"
)

print("Labelled articles:", len(labelled))
print("\nLabel distribution:")
print(labelled[label_cols].sum().rename(lambda x: x.replace("label_", "")))

# ----------------------------------------------------------------
# 2. Train/test split — 80% train, 20% test
#
# We hold out 20% of labelled articles as a test set that the
# model never sees during training. This gives an honest estimate
# of how well the model generalises to unseen articles.
# ----------------------------------------------------------------

X_text = labelled["content_processed"].fillna("")
Y      = labelled[label_cols].values

X_train_text, X_test_text, Y_train, Y_test = train_test_split(
    X_text, Y, test_size=0.2, random_state=42
)

print(f"\nTraining set : {len(X_train_text)} articles")
print(f"Test set     : {len(X_test_text)} articles")

# ----------------------------------------------------------------
# 3. TF-IDF — reduced features to prevent overfitting
#
# Reducing max_features from 5000 to 2000 limits the model's
# ability to memorise rare article-specific words.
# min_df=3 drops tokens appearing in fewer than 3 articles.
# ----------------------------------------------------------------

tfidf = TfidfVectorizer(max_features=2000, ngram_range=(1, 2), min_df=3)
X_train = tfidf.fit_transform(X_train_text)
X_test  = tfidf.transform(X_test_text)

print("\nTF-IDF feature matrix shape (train):", X_train.shape)

# ----------------------------------------------------------------
# 4. Train models with class balancing
# ----------------------------------------------------------------

lr_model = OneVsRestClassifier(
    LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced", C=0.5)
    # C=0.5 adds stronger regularisation to reduce overfitting
)
rf_model = OneVsRestClassifier(
    RandomForestClassifier(n_estimators=100, random_state=42,
                           class_weight="balanced_subsample", max_depth=10)
    # max_depth=10 prevents trees from growing deep enough to memorise training data
)

# 5-fold cross-validation on training set only
print("\nCross-validating on training set (5-fold)...")
lr_cv = cross_val_score(lr_model, X_train, Y_train, cv=5, scoring="f1_weighted")
rf_cv = cross_val_score(rf_model, X_train, Y_train, cv=5, scoring="f1_weighted")

print(f"Logistic Regression CV F1: {lr_cv.mean():.3f} ± {lr_cv.std():.3f}")
print(f"Random Forest       CV F1: {rf_cv.mean():.3f} ± {rf_cv.std():.3f}")

# ----------------------------------------------------------------
# 5. Evaluate on held-out test set
#    This is the honest performance metric for your report
# ----------------------------------------------------------------

lr_model.fit(X_train, Y_train)
rf_model.fit(X_train, Y_train)

lr_preds = lr_model.predict(X_test)
rf_preds = rf_model.predict(X_test)

label_names = [c.replace("label_", "") for c in label_cols]

print("\nLogistic Regression — Test Set Report:")
print(classification_report(Y_test, lr_preds, target_names=label_names, zero_division=0))

print("Random Forest — Test Set Report:")
print(classification_report(Y_test, rf_preds, target_names=label_names, zero_division=0))

print(f"Hamming Loss — Logistic Regression : {hamming_loss(Y_test, lr_preds):.4f}")
print(f"Hamming Loss — Random Forest       : {hamming_loss(Y_test, rf_preds):.4f}")

# ----------------------------------------------------------------
# 6. Retrain best model on full labelled set and predict all articles
#
# After honest evaluation, retrain on all 194 labelled articles
# before applying to the full 1409 article corpus.
# ----------------------------------------------------------------

# Retrain tfidf and models on full labelled data
tfidf_full = TfidfVectorizer(max_features=2000, ngram_range=(1, 2), min_df=3)
X_full     = tfidf_full.fit_transform(X_text)

lr_model.fit(X_full, Y)
rf_model.fit(X_full, Y)

# Apply to all 1409 articles
X_all     = tfidf_full.transform(news["content_processed"].fillna(""))
lr_all    = lr_model.predict(X_all)
rf_all    = rf_model.predict(X_all)

# Use Logistic Regression as final model if CV F1 was higher, else Random Forest
# (update this based on your results above)
final_preds = lr_all  # change to rf_all if RF performed better

pred_df = pd.DataFrame(final_preds, columns=[c.replace("label_", "pred_") for c in label_cols])
news    = pd.concat([news.reset_index(drop=True), pred_df], axis=1)

print("\nPredicted label distribution across all 1409 articles:")
pred_cols = [c for c in news.columns if c.startswith("pred_")]
print(news[pred_cols].sum().rename(lambda x: x.replace("pred_", "")))

# ----------------------------------------------------------------
# 7. Save outputs
# ----------------------------------------------------------------

joblib.dump(lr_model,    "model_logistic_regression.pkl")
joblib.dump(rf_model,    "model_random_forest.pkl")
joblib.dump(tfidf_full,  "tfidf_vectorizer.pkl")

news.to_csv("stage4_news_predictions.csv", index=False, encoding="utf-8-sig")

print("\nStage 4 done.")
print("  stage4_news_predictions.csv — articles with predicted risk labels")
print("  model_*.pkl / tfidf_vectorizer.pkl — saved for Stage 5")

Labelled articles: 194

Label distribution:
child_labour               51
forced_labour             155
discrimination            151
freedom_of_association     82
working_hours             187
compensation              173
osh                       166
contracts                  74
dtype: int64

Training set : 155 articles
Test set     : 39 articles

TF-IDF feature matrix shape (train): (155, 2000)

Cross-validating on training set (5-fold)...
Logistic Regression CV F1: 0.827 ± 0.022
Random Forest       CV F1: 0.796 ± 0.009

Logistic Regression — Test Set Report:
                        precision    recall  f1-score   support

          child_labour       0.44      0.62      0.52        13
         forced_labour       0.82      0.90      0.86        30
        discrimination       0.78      1.00      0.88        28
freedom_of_association       0.75      0.41      0.53        22
         working_hours       0.97      1.00      0.99        38
          compensation       0.95      1.00 

In [5]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import classification_report, hamming_loss
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

news  = pd.read_csv("stage3_news_features.csv")
bhrrc = pd.read_csv("stage1_bhrrc_labels.csv")

label_cols = [
    "label_child_labour", "label_forced_labour", "label_discrimination",
    "label_freedom_of_association", "label_working_hours",
    "label_compensation", "label_osh", "label_contracts"
]

news["factory_ref"] = news["osh_factory_matched"].fillna(news["bhrrc_factory_matched"])
labelled = news[news["factory_ref"].notna()].copy()
bhrrc_labels = bhrrc[["factory_name"] + label_cols].copy()
bhrrc_labels["factory_lower"] = bhrrc_labels["factory_name"].str.lower().str.strip()
labelled["factory_ref_lower"] = labelled["factory_ref"].str.lower().str.strip()
labelled = labelled.merge(bhrrc_labels.drop(columns=["factory_name"]),
                          left_on="factory_ref_lower", right_on="factory_lower", how="inner")

X_text = labelled["content_processed"].fillna("")
Y      = labelled[label_cols].values

X_train_text, X_test_text, Y_train, Y_test = train_test_split(
    X_text, Y, test_size=0.2, random_state=42
)

tfidf   = TfidfVectorizer(max_features=2000, ngram_range=(1, 2), min_df=3)
X_train = tfidf.fit_transform(X_train_text)
X_test  = tfidf.transform(X_test_text)

lr_model = joblib.load("model_logistic_regression.pkl")
rf_model = joblib.load("model_random_forest.pkl")

lr_model.fit(X_train, Y_train)
rf_model.fit(X_train, Y_train)

lr_preds = lr_model.predict(X_test)
rf_preds = rf_model.predict(X_test)

label_names = [c.replace("label_", "") for c in label_cols]

pd.set_option("display.width", 120)

print("Logistic Regression — Test Set Report:")
print(classification_report(Y_test, lr_preds, target_names=label_names, zero_division=0))
print(f"Hamming Loss: {hamming_loss(Y_test, lr_preds):.4f}")

print("\nRandom Forest — Test Set Report:")
print(classification_report(Y_test, rf_preds, target_names=label_names, zero_division=0))
print(f"Hamming Loss: {hamming_loss(Y_test, rf_preds):.4f}")

Logistic Regression — Test Set Report:
                        precision    recall  f1-score   support

          child_labour       0.44      0.62      0.52        13
         forced_labour       0.82      0.90      0.86        30
        discrimination       0.78      1.00      0.88        28
freedom_of_association       0.75      0.41      0.53        22
         working_hours       0.97      1.00      0.99        38
          compensation       0.95      1.00      0.97        36
                   osh       0.89      1.00      0.94        34
             contracts       0.59      0.62      0.61        16

             micro avg       0.82      0.88      0.85       217
             macro avg       0.77      0.82      0.79       217
          weighted avg       0.83      0.88      0.84       217
           samples avg       0.81      0.88      0.83       217

Hamming Loss: 0.2179

Random Forest — Test Set Report:
                        precision    recall  f1-score   support

      

"Logistic Regression with class balancing outperformed Random Forest on the test set (weighted F1: 0.84 vs 0.79). Performance varied across risk dimensions, with working hours (F1=0.99) and compensation (F1=0.97) achieving near-perfect scores due to distinctive vocabulary, while child labour (F1=0.52) and freedom of association (F1=0.53) scored lower due to limited training examples and subtler linguistic patterns. These limitations are consistent with the known challenges of multi-label classification on small labelled datasets."